# Supplemental code for Lesson 3

```bash
pip install numpy scikit-learn
```


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_auc_score, roc_curve,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv("Titanic-Dataset.csv")
df.head()

In [ ]:
df_clean = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())
df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])

df_clean["FamilySize"] = df_clean["SibSp"] + df_clean["Parch"] + 1
df_clean["IsAlone"] = (df_clean["FamilySize"] == 1).astype(int)
df_clean["LogFare"] = np.log1p(df_clean["Fare"])

df_clean["Sex"] = df_clean["Sex"].map({"male": 0, "female": 1})
df_clean["Embarked"] = df_clean["Embarked"].map({"S": 0, "C": 1, "Q": 2})

df_clean = df_clean.drop(columns=["SibSp", "Parch", "Fare"])

print("Missing values remaining:", df_clean.isnull().sum().sum())
print("Shape after prep:", df_clean.shape)
df_clean.head()

In [ ]:
FEATURES = ["Pclass", "Sex", "Age", "LogFare", "Embarked", "FamilySize", "IsAlone"]
TARGET = "Survived"

X = df_clean[FEATURES]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set:  {X_train.shape[0]} rows")
print(f"Test set:      {X_test.shape[0]} rows")
print(f"\nSurvival rate in train: {y_train.mean():.3f}")
print(f"Survival rate in test:  {y_test.mean():.3f}")

In [ ]:
majority_class = y_train.mode()[0]
naive_preds = np.full(len(y_test), majority_class)
naive_accuracy = accuracy_score(y_test, naive_preds)

print(f"Majority class: {majority_class} (Did not survive)")
print(f"Naive baseline accuracy: {naive_accuracy:.3f}")
print()
print("Interpretation: A model that always predicts 'did not survive'")
print(f"would be correct {naive_accuracy*100:.1f}% of the time — without learning anything.")
print(f"Our model must beat {naive_accuracy:.3f} to be worth using.")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  

log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

print("Model trained successfully.")

In [ ]:
y_pred  = log_reg.predict(X_test_scaled)
y_proba = log_reg.predict_proba(X_test_scaled)[:, 1] 

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("=" * 40)
print(f"  Naive Baseline Accuracy : {naive_accuracy:.3f}")
print(f"  Logistic Regression Acc : {acc:.3f}  (+{acc - naive_accuracy:.3f} vs baseline)")
print(f"  ROC-AUC Score           : {auc:.3f}")
print("=" * 40)
print(classification_report(y_test, y_pred, target_names=["Did Not Survive", "Survived"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Did Not Survive", "Survived"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix", fontsize=13, fontweight="bold")

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color="steelblue", linewidth=2, label=f"Logistic Reg (AUC = {auc:.3f})")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Baseline (AUC = 0.5)")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve", fontsize=13, fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
coef_df = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": log_reg.coef_[0]
}).sort_values("Coefficient")

colors = ["#e05c5c" if c < 0 else "#4a90a4" for c in coef_df["Coefficient"]]

plt.figure(figsize=(8, 5))
plt.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Logistic Regression Coefficients\n(Blue = increases survival odds, Red = decreases)",
          fontsize=12, fontweight="bold")
plt.xlabel("Coefficient Value (after scaling)")
plt.tight_layout()
plt.show()

print(coef_df.to_string(index=False))

In [ ]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42, max_iter=1000))
])
cv_scores = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")

print("5-Fold Cross-Validation Accuracy:")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.3f}")
print(f"\n  Mean:  {cv_scores.mean():.3f}")
print(f"  Std:   {cv_scores.std():.3f}")
print()
print("A low std (< 0.02) means the model is stable across different data splits.")

In [ ]:
import joblib
joblib.dump(pipe, "titanic_logistic_model.pkl")

# Part 2 — Linear Regression on House Prices

Go to this comp and join the comp:
kaggle.com/c/house-prices-advanced-regression-techniques

```bash
kaggle competitions download -c house-prices-advanced-regression-techniques
unzip house-prices-advanced-regression-techniques.zip
```

In [ ]:
houses = pd.read_csv("train.csv") 
houses.head()

In [ ]:

print("Shape:", houses.shape)
print("\nTarget — SalePrice:")
print(houses["SalePrice"].describe().apply(lambda x: f"${x:,.0f}"))

In [ ]:
pd.set_option('display.max_rows', None)
eda_summary = pd.DataFrame({
    "dtype": houses.dtypes,
    "num_unique": houses.nunique(),
    "num_missing": houses.isnull().sum(),
    "missing_pct": houses.isnull().mean() * 100
})

eda_summary.sort_values("num_unique")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(houses["SalePrice"], bins=50, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("SalePrice Distribution (Original)")
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))

sns.histplot(np.log1p(houses["SalePrice"]), bins=50, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("SalePrice Distribution (Log Transformed)")

plt.suptitle("House Sale Price — Before and After Log Transform", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Original skew:          {houses['SalePrice'].skew():.3f}")
print(f"Log-transformed skew:   {np.log1p(houses['SalePrice']).skew():.3f}")
print()
print("Linear regression assumes the target is roughly normally distributed.")
print("High skew violates that assumption — log transform fixes it.")

In [ ]:
HOUSE_FEATURES = [
    "GrLivArea",      
    "OverallQual",    
    "GarageCars",     
    "TotalBsmtSF",    
    "FullBath",       
    "YearBuilt",      
    "TotRmsAbvGrd",   
]

houses_clean = houses[HOUSE_FEATURES + ["SalePrice"]].dropna()

houses_clean["LogSalePrice"] = np.log1p(houses_clean["SalePrice"])

print(f"Rows after dropping any missing: {len(houses_clean)}")
houses_clean[HOUSE_FEATURES].describe().round(1)


In [ ]:
import seaborn as sns
corr_with_price = houses_clean[HOUSE_FEATURES + ["SalePrice"]].corr()["SalePrice"].drop("SalePrice").sort_values(ascending=False)

corr_df = corr_with_price.reset_index()
corr_df.columns = ["Feature", "Correlation"]

plt.figure(figsize=(7, 4))
sns.barplot(data=corr_df, x="Correlation", y="Feature", palette="coolwarm")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlation of Each Feature with SalePrice")
plt.tight_layout()
plt.show()

In [ ]:
X_h = houses_clean[HOUSE_FEATURES]
y_h = houses_clean["LogSalePrice"]

X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)


mean_log_price = y_h_train.mean()
naive_preds_h  = np.full(len(y_h_test), mean_log_price)

naive_rmse = np.sqrt(mean_squared_error(y_h_test, naive_preds_h))
naive_rmse_dollars = np.expm1(mean_log_price + naive_rmse) - np.expm1(mean_log_price)

print(f"Naive baseline (predict mean log price):")
print(f"  RMSE (log scale):  {naive_rmse:.4f}")
print(f"  Approximate error in dollars: ${naive_rmse_dollars:,.0f}")


scaler_h = StandardScaler()
X_h_train_scaled = scaler_h.fit_transform(X_h_train)
X_h_test_scaled  = scaler_h.transform(X_h_test)

lin_reg = LinearRegression()
lin_reg.fit(X_h_train_scaled, y_h_train)

print("\nLinear regression trained.")

In [ ]:
y_h_pred = lin_reg.predict(X_h_test_scaled)

rmse = np.sqrt(mean_squared_error(y_h_test, y_h_pred))
mae  = mean_absolute_error(y_h_test, y_h_pred)
r2   = r2_score(y_h_test, y_h_pred)

y_h_test_dollars = np.expm1(y_h_test)
y_h_pred_dollars = np.expm1(y_h_pred)
rmse_dollars = np.sqrt(mean_squared_error(y_h_test_dollars, y_h_pred_dollars))

print("=" * 50)
print(f"  Naive Baseline RMSE (log): {naive_rmse:.4f}")
print(f"  Linear Regression RMSE  : {rmse:.4f}  ({'-' if rmse < naive_rmse else '+'}{ abs(rmse - naive_rmse):.4f} vs baseline)")
print()
print(f"  MAE  (log scale): {mae:.4f}")
print(f"  R²   Score:       {r2:.3f}")
print("=" * 50)
print()
print("R² interpretation:")
print(f"  {r2:.3f} means the model explains {r2*100:.1f}% of the variance in house prices")
print("  R² of 1.0 = perfect. R² of 0.0 = as good as predicting the mean (our baseline).")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

residuals = y_h_test - y_h_pred

sns.scatterplot(x=y_h_test, y=y_h_pred, alpha=0.4, ax=axes[0])
min_val, max_val = y_h_test.min(), y_h_test.max()
axes[0].plot([min_val, max_val], [min_val, max_val], color="red", linewidth=1.5, linestyle="--", label="Perfect prediction")
axes[0].set_xlabel("Actual Log SalePrice")
axes[0].set_ylabel("Predicted Log SalePrice")
axes[0].set_title("Actual vs Predicted")
axes[0].legend()

sns.scatterplot(x=y_h_pred, y=residuals, alpha=0.4, ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1.2, linestyle="--")
axes[1].set_xlabel("Predicted Log SalePrice")
axes[1].set_ylabel("Residual (Actual − Predicted)")
axes[1].set_title("Residual Plot")

plt.suptitle("Linear Regression — Diagnostic Plots", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
coef_h_df = pd.DataFrame({
    "Feature": HOUSE_FEATURES,
    "Coefficient": lin_reg.coef_
}).sort_values("Coefficient")

plt.figure(figsize=(8, 4))
sns.barplot(data=coef_h_df, x="Coefficient", y="Feature", palette="coolwarm")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Linear Regression Coefficients\n(After scaling — larger bar = stronger influence on log price)")
plt.xlabel("Coefficient Value")
plt.tight_layout()
plt.show()

print(coef_h_df.to_string(index=False))
print(f"\nIntercept: {lin_reg.intercept_:.4f}")

In [ ]:
test = pd.read_csv("test.csv")
print(f"Test set shape: {test.shape}")

# Grab the same features we used for training
HOUSE_FEATURES = [
    "GrLivArea",      
    "OverallQual",    
    "GarageCars",     
    "TotalBsmtSF",    
    "FullBath",       
    "YearBuilt",      
    "TotRmsAbvGrd",   
]
X_test_submission = test[HOUSE_FEATURES].copy()
X_test_submission = X_test_submission.fillna(X_test_submission.median())

print(f"\nMissing values in test features:")
print(X_test_submission.isnull().sum())

X_test_submission_scaled = scaler_h.transform(X_test_submission)
log_predictions = lin_reg.predict(X_test_submission_scaled)
predicted_prices = np.expm1(log_predictions)

submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': predicted_prices
})

# Save to CSV for Kaggle submission
submission.to_csv('house_prices_submission.csv', index=False)

print(f"\n{'='*60}")
print(f"Predictions generated for {len(submission)} houses!")
print(f"{'='*60}")
print(f"\nPredicted Price Statistics:")
print(f"  Minimum:  ${predicted_prices.min():>12,.0f}")
print(f"  Median:   ${np.median(predicted_prices):>12,.0f}")
print(f"  Mean:     ${predicted_prices.mean():>12,.0f}")
print(f"  Maximum:  ${predicted_prices.max():>12,.0f}")
print(f"\nFirst 10 predictions:")
print(submission.head(10).to_string(index=False))
print(f"\n✓ Submission file saved as 'house_prices_submission.csv'")

# Submit to kaggle
```bash
kaggle competitions submit -c house-prices-advanced-regression-techniques -f house_prices_submission.csv -m "My submission"
```

# End of class acitivty - improve the current system by adding more relevant features

In [ ]:
houses = pd.read_csv("train.csv")
print(houses.shape)

numeric_cols = houses.select_dtypes(include="number").drop(columns=["Id"]).columns
corr_all = houses[numeric_cols].corr()[["SalePrice"]].drop("SalePrice").sort_values("SalePrice", ascending=False)

plt.figure(figsize=(4, 14))
sns.heatmap(corr_all, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("All Numeric Features — Correlation with SalePrice")
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = houses.select_dtypes(include="object").columns.tolist()

fig, axes = plt.subplots(len(cat_cols)//3 + 1, 3, figsize=(18, 60))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    # Sort by median price so the plot is easy to read
    order = houses.groupby(col)["SalePrice"].median().sort_values(ascending=False).index
    sns.boxplot(data=houses, x=col, y="SalePrice", order=order, ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel("")
    axes[i].tick_params(axis="x", rotation=45)

# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Categorical Features — SalePrice Distribution", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()